# Transformer-based rPPG Pipeline for Heart Rate Estimation

A complete deep learning pipeline using Transformer architecture to estimate heart rate from facial videos.

## Overview
- **Dataset**: UBFC-rPPG
- **Face Detection**: MediaPipe/OpenCV Haar Cascade
- **ROI Extraction**: Forehead and cheeks
- **Signal Processing**: Bandpass filtering (0.7-4 Hz)
- **Model**: Transformer Encoder with Positional Encoding
- **Outputs**: rPPG signal (sequence) + BPM (scalar)
- **Loss**: Combined MSE + MAE + FFT-based frequency loss

## Table of Contents
1. [Installation and Setup](#Installation)
2. [Configuration](#Configuration)
3. [Utility Modules](#Utilities)
4. [Dataset Loader](#Dataset)
5. [Model Architecture](#Model)
6. [Loss Functions](#Loss)
7. [Training Loop](#Training)
8. [Evaluation](#Evaluation)
9. [Inference](#Inference)

## 1. Installation and Setup

In [ ]:
!pip install torch torchvision numpy opencv-python mediapipe scipy scikit-learn matplotlib tqdm pandas seaborn

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import os
import sys
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
import scipy.io

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class Config:
    # Dataset
    data_root: str = "/home/dhruv/Documents/rppg project/UBFC/dataset 2"
    sequence_length: int = 300
    fps: int = 30
    ppg_fs: int = 60
    batch_size: int = 4
    num_workers: int = 0
    train_split: float = 0.8
    
    # ROI Extraction
    roi_regions: List[str] = None
    face_detection_confidence: float = 0.5
    
    # Signal Processing
    bandpass_low: float = 0.7
    bandpass_high: float = 4.0
    
    # Model
    d_model: int = 128
    nhead: int = 8
    num_encoder_layers: int = 6
    dim_feedforward: int = 512
    dropout: float = 0.1
    max_seq_length: int = 300
    
    # Training
    num_epochs: int = 10
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    signal_loss_weight: float = 1.0
    bpm_loss_weight: float = 0.5
    freq_loss_weight: float = 0.3
    
    # Optimization
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "checkpoints"
    model_name: str = "transformer_rppg.pth"
    
    def __post_init__(self):
        if self.roi_regions is None:
            self.roi_regions = ["forehead", "left_cheek", "right_cheek"]
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()
print(f"Device: {config.device}")
print(f"Data root: {config.data_root}")

## 3. Utility Modules

### 3.1 Face Detection

In [ ]:
class FaceDetector:
    def __init__(self, min_detection_confidence: float = 0.5):
        self.face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        )
        
        # Try MediaPipe
        self.use_mediapipe = False
        try:
            import mediapipe as mp
            self.mp_face_detection = mp.solutions.face_detection
            self.face_detection = self.mp_face_detection.FaceDetection(
                min_detection_confidence=min_detection_confidence
            )
            self.use_mediapipe = True
            print("Using MediaPipe for face detection")
        except (ImportError, AttributeError):
            print("Using OpenCV Haar Cascade")
    
    def detect(self, frame):
        if self.use_mediapipe:
            return self._detect_mediapipe(frame)
        else:
            return self._detect_opencv(frame)
    
    def _detect_mediapipe(self, frame):
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.face_detection.process(rgb_frame)
        if results.detections:
            detection = results.detections[0]
            bbox = detection.location_data.relative_bounding_box
            h, w = frame.shape[:2]
            x = int(bbox.xmin * w)
            y = int(bbox.ymin * h)
            width = int(bbox.width * w)
            height = int(bbox.height * h)
            return (x, y, width, height)
        return None
    
    def _detect_opencv(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = self.face_cascade.detectMultiScale(gray, 1.3, 5)
        if len(faces) > 0:
            x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
            return (x, y, w, h)
        return None
    
    def close(self):
        if self.use_mediapipe:
            self.face_detection.close()

# Test face detector
detector = FaceDetector()
print("Face detector initialized!")

### 3.2 ROI Extraction

In [ ]:
class ROIExtractor:
    def __init__(self, regions=None):
        self.regions = regions or ["forehead", "left_cheek", "right_cheek"]
    
    def extract_roi(self, frame, bbox):
        x, y, w, h = bbox
        roi_regions = {}
        
        if "forehead" in self.regions:
            forehead_y = y
            forehead_h = int(h * 0.15)
            forehead_x = x + int(w * 0.25)
            forehead_w = int(w * 0.5)
            roi_regions["forehead"] = frame[
                forehead_y:forehead_y + forehead_h,
                forehead_x:forehead_x + forehead_w
            ]
        
        if "left_cheek" in self.regions:
            left_cheek_x = x
            left_cheek_y = y + int(h * 0.4)
            left_cheek_w = int(w * 0.3)
            left_cheek_h = int(h * 0.3)
            roi_regions["left_cheek"] = frame[
                left_cheek_y:left_cheek_y + left_cheek_h,
                left_cheek_x:left_cheek_x + left_cheek_w
            ]
        
        if "right_cheek" in self.regions:
            right_cheek_x = x + int(w * 0.7)
            right_cheek_y = y + int(h * 0.4)
            right_cheek_w = int(w * 0.3)
            right_cheek_h = int(h * 0.3)
            roi_regions["right_cheek"] = frame[
                right_cheek_y:right_cheek_y + right_cheek_h,
                right_cheek_x:right_cheek_x + right_cheek_w
            ]
        
        return roi_regions
    
    def compute_mean_rgb(self, roi_regions):
        rgb_values = []
        for region_name, roi in roi_regions.items():
            if roi.size > 0:
                mean_rgb = np.mean(roi, axis=(0, 1))
                rgb_values.append(mean_rgb)
        if rgb_values:
            return np.mean(rgb_values, axis=0)
        return np.zeros(3)

print("ROI Extractor defined!")

### 3.3 Signal Processing

In [ ]:
class SignalProcessor:
    def __init__(self, fs=30, lowcut=0.7, highcut=4.0):
        self.fs = fs
        self.lowcut = lowcut
        self.highcut = highcut
    
    def bandpass_filter(self, signal_data):
        nyquist = 0.5 * self.fs
        low = self.lowcut / nyquist
        high = self.highcut / nyquist
        b, a = sp_signal.butter(4, [low, high], btype='band')
        filtered = sp_signal.filtfilt(b, a, signal_data, axis=0)
        return filtered
    
    def compute_bpm_from_fft(self, signal_data):
        n = len(signal_data)
        if n == 0:
            return 0.0
        yf = np.fft.fft(signal_data)
        xf = np.fft.fftfreq(n, 1 / self.fs)
        mask = (xf >= self.lowcut) & (xf <= self.highcut) & (xf > 0)
        freqs = xf[mask]
        magnitudes = np.abs(yf[mask])
        if len(magnitudes) > 0:
            peak_freq = freqs[np.argmax(magnitudes)]
            return float(peak_freq * 60)
        return 0.0
    
    def normalize_signal(self, signal_data):
        mean = np.mean(signal_data, axis=0)
        std = np.std(signal_data, axis=0)
        std[std == 0] = 1.0
        return (signal_data - mean) / std

print("Signal Processor defined!")

## 4. Dataset Loader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class UBFCDataset(Dataset):
    def __init__(self, data_root, sequence_length=300, fps=30, ppg_fs=60, is_train=True, split_ratio=0.8):
        self.data_root = data_root
        self.sequence_length = sequence_length
        self.fps = fps
        self.ppg_fs = ppg_fs
        self.is_train = is_train
        
        all_samples = self._load_samples()
        split_idx = int(len(all_samples) * split_ratio)
        if is_train:
            self.samples = all_samples[:split_idx]
        else:
            self.samples = all_samples[split_idx:]
        
        print(f"{'Train' if is_train else 'Val'} samples: {len(self.samples)}")
    
    def _load_samples(self):
        samples = []
        if not os.path.exists(self.data_root):
            print(f"WARNING: Data root {self.data_root} does not exist!")
            return samples
        
        for subject_dir in sorted(os.listdir(self.data_root)):
            subject_path = os.path.join(self.data_root, subject_dir)
            if not os.path.isdir(subject_path):
                continue
            
            video_path = None
            for vid_name in ["vid.avi", "video.avi"]:
                path = os.path.join(subject_path, vid_name)
                if os.path.exists(path):
                    video_path = path
                    break
            
            gt_path = None
            for gt_name in ["ground_truth.mat", "bvp.mat"]:
                path = os.path.join(subject_path, gt_name)
                if os.path.exists(path):
                    gt_path = path
                    break
            
            if video_path and gt_path:
                samples.append({'subject_id': subject_dir, 'video_path': video_path, 'gt_path': gt_path})
        
        print(f"Total samples found: {len(samples)}")
        return samples
    
    def _extract_rgb_signal(self, video_path):
        face_detector = FaceDetector()
        roi_extractor = ROIExtractor()
        signal_processor = SignalProcessor(fs=self.fps)
        
        cap = cv2.VideoCapture(video_path)
        rgb_signals = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            bbox = face_detector.detect(frame)
            if bbox:
                roi_regions = roi_extractor.extract_roi(frame, bbox)
                mean_rgb = roi_extractor.compute_mean_rgb(roi_regions)
                rgb_signals.append(mean_rgb)
        cap.release()
        face_detector.close()
        
        if len(rgb_signals) == 0:
            return np.zeros((self.sequence_length, 3))
        
        rgb_signals = np.array(rgb_signals)
        rgb_signals = signal_processor.bandpass_filter(rgb_signals)
        rgb_signals = signal_processor.normalize_signal(rgb_signals)
        return rgb_signals
    
    def _load_ground_truth(self, gt_path, target_length):
        if gt_path.endswith('.mat'):
            mat_data = scipy.io.loadmat(gt_path)
            signal = None
            for key in ['bvp', 'BVP', 'ppg', 'PPG']:
                if key in mat_data:
                    signal = mat_data[key].flatten()
                    break
            if signal is None:
                return np.zeros(target_length)
        else:
            return np.zeros(target_length)
        
        if len(signal) != target_length:
            signal = sp_signal.resample(signal, target_length)
        return signal
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        video_path = sample['video_path']
        gt_path = sample['gt_path']
        
        rgb_signal = self._extract_rgb_signal(video_path)
        min_len = min(len(rgb_signal), self.sequence_length * 2)
        gt_ppg = self._load_ground_truth(gt_path, min_len)
        
        if len(rgb_signal) >= self.sequence_length:
            if self.is_train:
                start = np.random.randint(0, len(rgb_signal) - self.sequence_length)
            else:
                start = 0
            rgb_signal = rgb_signal[start:start + self.sequence_length]
            gt_signal = sp_signal.resample(gt_ppg, self.sequence_length) if len(gt_ppg) > 0 else np.zeros(self.sequence_length)
        else:
            pad_len = self.sequence_length - len(rgb_signal)
            rgb_signal = np.pad(rgb_signal, ((0, pad_len), (0, 0)), mode='constant')
            gt_signal = sp_signal.resample(gt_ppg, self.sequence_length) if len(gt_ppg) > 0 else np.zeros(self.sequence_length)
        
        rgb_tensor = torch.FloatTensor(rgb_signal)
        gt_signal_tensor = torch.FloatTensor(gt_signal)
        
        sp = SignalProcessor(fs=self.fps)
        gt_bpm = sp.compute_bpm_from_fft(gt_signal)
        gt_bpm_tensor = torch.FloatTensor([gt_bpm])
        
        return rgb_tensor, gt_signal_tensor, gt_bpm_tensor

print("UBFCDataset class defined!")

In [ ]:
# Test dataset loading
dataset = UBFCDataset(
    data_root=config.data_root,
    sequence_length=config.sequence_length,
    fps=config.fps,
    is_train=True
)

if len(dataset) > 0:
    sample = dataset[0]
    print(f"RGB signal shape: {sample[0].shape}")
    print(f"GT signal shape: {sample[1].shape}")
    print(f"GT BPM: {sample[2].item():.2f}")

## 5. Model Architecture

In [ ]:
import math
from typing import Tuple

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerEncoder(nn.Module):
    def __init__(self, input_dim=3, d_model=128, nhead=8, num_encoder_layers=6, dim_feedforward=512, dropout=0.1, max_seq_length=300):
        super().__init__()
        self.d_model = d_model
        self.input_projection = nn.Linear(input_dim, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)
        self.signal_head = nn.Linear(d_model, 1)
        self.bpm_head = nn.Linear(d_model, 1)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = self.input_projection(x)
        x = self.dropout(x)
        x = self.positional_encoding(x)
        encoded = self.transformer_encoder(x)
        rppg_signal = self.signal_head(encoded).squeeze(-1)
        pooled = encoded.mean(dim=1)
        bpm = self.bpm_head(pooled)
        return rppg_signal, bpm

# Test model
model = TransformerEncoder(
    input_dim=3,
    d_model=config.d_model,
    nhead=config.nhead,
    num_encoder_layers=config.num_encoder_layers,
    dim_feedforward=config.dim_feedforward,
    dropout=config.dropout,
    max_seq_length=config.max_seq_length
).to(config.device)

dummy_input = torch.randn(4, config.sequence_length, 3).to(config.device)
pred_signal, pred_bpm = model(dummy_input)
print(f"Model test passed!")
print(f"Pred signal shape: {pred_signal.shape}")
print(f"Pred BPM shape: {pred_bpm.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

## 6. Loss Functions

In [ ]:
class RPPGLoss(nn.Module):
    def __init__(self, signal_weight=1.0, bpm_weight=0.5, freq_weight=0.3):
        super().__init__()
        self.signal_weight = signal_weight
        self.bpm_weight = bpm_weight
        self.freq_weight = freq_weight
        self.mse_loss = nn.MSELoss()
        self.mae_loss = nn.L1Loss()
    
    def frequency_loss(self, pred, target, fs=30):
        pred_fft = torch.fft.rfft(pred, dim=-1)
        target_fft = torch.fft.rfft(target, dim=-1)
        pred_mag = torch.abs(pred_fft)
        target_mag = torch.abs(target_fft)
        freqs = torch.fft.rfftfreq(pred.size(-1), d=1/fs).to(pred.device)
        mask = (freqs >= 0.7) & (freqs <= 4.0)
        pred_masked = pred_mag * mask.to(pred_mag.device)
        target_masked = target_mag * mask.to(target_mag.device)
        return self.mse_loss(pred_masked, target_masked)
    
    def forward(self, pred_signal, pred_bpm, target_signal, target_bpm, fs=30):
        signal_loss = self.mse_loss(pred_signal, target_signal)
        bpm_loss = self.mae_loss(pred_bpm, target_bpm)
        freq_loss = self.frequency_loss(pred_signal, target_signal, fs)
        total_loss = (
            self.signal_weight * signal_loss +
            self.bpm_weight * bpm_loss +
            self.freq_weight * freq_loss
        )
        loss_dict = {
            'total_loss': total_loss.item(),
            'signal_loss': signal_loss.item(),
            'bpm_loss': bpm_loss.item(),
            'freq_loss': freq_loss.item()
        }
        return total_loss, loss_dict

# Test loss
criterion = RPPGLoss()
target_signal = torch.randn_like(pred_signal)
target_bpm = torch.randn(4, 1).to(config.device)
loss, loss_dict = criterion(pred_signal, pred_bpm, target_signal, target_bpm, config.fps)
print(f"Loss test passed! Loss: {loss.item():.4f}")

## 7. Training Loop

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    loss_dicts = []
    num_batches = 0
    
    for batch_idx, (rgb_signal, gt_signal, gt_bpm) in enumerate(tqdm(dataloader, desc="Training")):
        if rgb_signal is None:
            continue
        rgb_signal = rgb_signal.to(device)
        gt_signal = gt_signal.to(device)
        gt_bpm = gt_bpm.to(device)
        
        optimizer.zero_grad()
        pred_signal, pred_bpm = model(rgb_signal)
        loss, loss_dict = criterion(pred_signal, pred_bpm, gt_signal, gt_bpm, config.fps)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        loss_dicts.append(loss_dict)
        num_batches += 1
    
    if num_batches == 0:
        return 0, {}
    avg_loss = total_loss / num_batches
    avg_loss_dict = {k: np.mean([d[k] for d in loss_dicts]) for k in loss_dicts[0].keys()}
    return avg_loss, avg_loss_dict

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    loss_dicts = []
    num_batches = 0
    
    with torch.no_grad():
        for batch_idx, (rgb_signal, gt_signal, gt_bpm) in enumerate(tqdm(dataloader, desc="Validation")):
            if rgb_signal is None:
                continue
            rgb_signal = rgb_signal.to(device)
            gt_signal = gt_signal.to(device)
            gt_bpm = gt_bpm.to(device)
            pred_signal, pred_bpm = model(rgb_signal)
            loss, loss_dict = criterion(pred_signal, pred_bpm, gt_signal, gt_bpm, config.fps)
            total_loss += loss.item()
            loss_dicts.append(loss_dict)
            num_batches += 1
    
    if num_batches == 0:
        return 0, {}
    avg_loss = total_loss / num_batches
    avg_loss_dict = {k: np.mean([d[k] for d in loss_dicts]) for k in loss_dicts[0].keys()}
    return avg_loss, avg_loss_dict

print("Training functions defined!")

In [ ]:
# Setup training
os.makedirs(config.save_dir, exist_ok=True)

train_dataset = UBFCDataset(
    data_root=config.data_root,
    sequence_length=config.sequence_length,
    fps=config.fps,
    is_train=True
)

val_dataset = UBFCDataset(
    data_root=config.data_root,
    sequence_length=config.sequence_length,
    fps=config.fps,
    is_train=False
)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=0)

model = TransformerEncoder(
    input_dim=3,
    d_model=config.d_model,
    nhead=config.nhead,
    num_encoder_layers=config.num_encoder_layers,
    dim_feedforward=config.dim_feedforward,
    dropout=config.dropout,
    max_seq_length=config.max_seq_length
).to(config.device)

criterion = RPPGLoss(
    signal_weight=config.signal_loss_weight,
    bpm_weight=config.bpm_loss_weight,
    freq_weight=config.freq_loss_weight
)

optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

print("Setup complete! Ready to train.")

In [ ]:
# Training loop
best_val_loss = float('inf')

for epoch in range(config.num_epochs):
    print(f"\nEpoch {epoch+1}/{config.num_epochs}")
    
    train_loss, train_loss_dict = train_epoch(model, train_loader, criterion, optimizer, config.device)
    val_loss, val_loss_dict = validate(model, val_loader, criterion, config.device)
    
    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    if train_loss_dict:
        print(f"Train Losses: {train_loss_dict}")
    
    if val_loss > 0:
        scheduler.step(val_loss)
    
    if 0 < val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, os.path.join(config.save_dir, config.model_name))
        print(f"Saved best model with val_loss: {val_loss:.4f}")

print("\nTraining complete!")

## 8. Evaluation

In [ ]:
from scipy.stats import pearsonr
from scipy.fft import fft, fftfreq

def compute_metrics(pred_bpm, gt_bpm):
    mae = np.mean(np.abs(pred_bpm - gt_bpm))
    rmse = np.sqrt(np.mean((pred_bpm - gt_bpm) ** 2))
    if len(pred_bpm) > 1:
        pearson, _ = pearsonr(pred_bpm, gt_bpm)
    else:
        pearson = 0.0
    return mae, rmse, pearson

def compute_bpm_from_signal(signal_data, fs=30):
    n = len(signal_data)
    if n == 0:
        return 0.0
    yf = fft(signal_data)
    xf = fftfreq(n, 1/fs)
    mask = (xf >= 0.7) & (xf <= 4.0) & (xf > 0)
    freqs = xf[mask]
    magnitudes = np.abs(yf[mask])
    if len(magnitudes) > 0:
        peak_freq = freqs[np.argmax(magnitudes)]
        return float(peak_freq * 60)
    return 0.0

def evaluate(model, dataloader, device, fs=30):
    model.eval()
    all_pred_bpm = []
    all_gt_bpm = []
    
    with torch.no_grad():
        for batch_idx, (rgb_signal, gt_signal, gt_bpm) in enumerate(tqdm(dataloader, desc="Evaluating")):
            if rgb_signal is None:
                continue
            rgb_signal = rgb_signal.to(device)
            gt_bpm_np = gt_bpm.numpy()
            pred_signal, pred_bpm = model(rgb_signal)
            pred_signal_np = pred_signal.cpu().numpy()
            for i in range(len(pred_signal_np)):
                bpm = compute_bpm_from_signal(pred_signal_np[i], fs)
                all_pred_bpm.append(bpm)
            all_gt_bpm.extend(gt_bpm_np.flatten())
    
    return np.array(all_pred_bpm), np.array(all_gt_bpm)

print("Evaluation functions defined!")

In [ ]:
# Run evaluation
model_path = os.path.join(config.save_dir, config.model_name)
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=config.device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded model from epoch {checkpoint['epoch']}")
    
    pred_bpm, gt_bpm = evaluate(model, val_loader, config.device, config.fps)
    mae, rmse, pearson = compute_metrics(pred_bpm, gt_bpm)
    
    print("\n" + "="*50)
    print("Evaluation Results:")
    print("="*50)
    print(f"MAE: {mae:.2f} BPM")
    print(f"RMSE: {rmse:.2f} BPM")
    print(f"Pearson Correlation: {pearson:.4f}")
    print("="*50)
else:
    print("No trained model found. Please train first.")

## 9. Inference

In [ ]:
class RPPGPredictor:
    def __init__(self, model_path, device=None):
        self.device = device or config.device
        self.model = TransformerEncoder(
            input_dim=3, d_model=config.d_model,
            nhead=config.nhead,
            num_encoder_layers=config.num_encoder_layers,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            max_seq_length=config.max_seq_length
        ).to(self.device)
        
        checkpoint = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        self.face_detector = FaceDetector()
        self.roi_extractor = ROIExtractor()
        self.signal_processor = SignalProcessor(fs=config.fps)
        print(f"Model loaded from {model_path}")
    
    def process_video(self, video_path):
        cap = cv2.VideoCapture(video_path)
        rgb_signals = []
        while True:
            ret, frame = cap.read()
            if not ret: break
            bbox = self.face_detector.detect(frame)
            if bbox:
                roi_regions = self.roi_extractor.extract_roi(frame, bbox)
                mean_rgb = self.roi_extractor.compute_mean_rgb(roi_regions)
                rgb_signals.append(mean_rgb)
        cap.release()
        self.face_detector.close()
        
        if len(rgb_signals) == 0:
            print("No face detected!")
            return None, None
        
        rgb_signals = np.array(rgb_signals)
        rgb_signals = self.signal_processor.bandpass_filter(rgb_signals)
        rgb_signals = self.signal_processor.normalize_signal(rgb_signals)
        
        if len(rgb_signals) < config.sequence_length:
            rgb_signals = np.pad(rgb_signals, ((0, config.sequence_length - len(rgb_signals)), (0, 0)), mode='constant')
        else:
            rgb_signals = rgb_signals[:config.sequence_length]
        
        rgb_tensor = torch.FloatTensor(rgb_signals).unsqueeze(0).to(self.device)
        with torch.no_grad():
            pred_signal, pred_bpm = self.model(rgb_tensor)
        
        return pred_signal.cpu().numpy().flatten(), pred_bpm.cpu().numpy().flatten()[0]

print("RPPGPredictor class defined!")

## Summary

This notebook implements a complete Transformer-based rPPG pipeline:

1. **Data Loading**: UBFC-rPPG dataset with video processing
2. **Preprocessing**: Face detection, ROI extraction, signal filtering
3. **Model**: Transformer encoder with positional encoding
4. **Training**: Multi-loss optimization (signal + BPM + frequency)
5. **Evaluation**: MAE, RMSE, Pearson correlation
6. **Inference**: Video processing for rPPG estimation

To use:
1. Update `config.data_root` with your dataset path
2. Run all cells to train the model
3. Use `RPPGPredictor` for inference on new videos